# LFW Grad-CAM — 02. Population Grad-CAM extraction

Pass B에서 LOO target이 있는 모든 선택 표본에 Grad-CAM을 계산합니다.
Pass A와 Pass B의 embedding cosine을 검사하고, 전체 표본 feature 행과
적격 표본의 native heatmap shard를 분리 저장합니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # arcface, adaface, magface 중 이번 실행 checkpoint
MODE = "real"               # 빠른 검증은 dev, 전체 논문 실행만 real
DATA_FRACTION = 1.0       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합과 random control의 재현 seed
EXECUTE_STAGE = True      # 필수 입력을 채우고 이 단계 계산 시에만 True
WRITE_OUTPUTS = True      # 새 immutable artifact 저장 시에만 True
OVERWRITE = True          # canonical stage result replacement

SELECTED_MODEL_FAMILIES = {
    CONFIG["models"]["profiles"][profile_name]["family"]
    for profile_name in CONFIG["models"]["selected_profiles"]
}
if MODEL_NAME not in SELECTED_MODEL_FAMILIES:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [ ]:
from research.runtime import RunStore, resolve_active_run

STEP2_RUN_DIR = resolve_active_run(
    PROJECT_ROOT / CONFIG["run"]["root"]
)
RUN = RunStore.open(STEP2_RUN_DIR)
WORKFLOW_ROOT = (
    STEP2_RUN_DIR / CONFIG["workflow"]["artifact_subdir"]
)

import numpy as np
import pandas as pd

from research.embeddings import (
    create_pytorch_adapter_from_spec,
    select_model_spec,
)
from research.explainability.gradcam import (
    extract_population_gradcam,
    read_prepared_population_artifact,
    write_population_saliency_artifact,
)
from research.runtime.hashing import sha256_file

PREPARED_ARTIFACT_DIR = WORKFLOW_ROOT / CONFIG["workflow"]["prepared_population_dir"]
SELECTED_MANIFEST_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["selected_manifest_path"]
ALIGNED_FACES_NPY_PATH = PROJECT_ROOT / CONFIG["aligned_crops"]["faces_path"]
MODEL_REGISTRY_ROOT = PROJECT_ROOT / "runs/step2/model_registry"
REGION_MASK_BUNDLE_PATH = None
SALIENCY_ARTIFACT_OUTPUT_DIR = WORKFLOW_ROOT / CONFIG["workflow"]["saliency_population_dir"]
DEVICE = "cuda"

RUN_FAITHFULNESS = True
CAPTURE_INTERMEDIATES = False  # activation/gradient 전량 영구저장 금지


In [ ]:
if EXECUTE_STAGE:
    required = {
        "prepared": PREPARED_ARTIFACT_DIR,
        "selected": SELECTED_MANIFEST_PATH,
        "aligned_faces": ALIGNED_FACES_NPY_PATH,
    }
    missing = [name for name, value in required.items() if value is None]
    if missing:
        raise RuntimeError(f"입력 경로가 비어 있습니다: {missing}")
    prepared = read_prepared_population_artifact(required["prepared"])
    selected = pd.read_csv(required["selected"])
    if not np.array_equal(
        selected["sample_id"].astype(str).to_numpy(),
        prepared.sample_ids.astype(str),
    ):
        raise ValueError("선택 manifest와 Pass A 표본 순서가 다릅니다.")
    source_faces = np.load(
        required["aligned_faces"],
        mmap_mode="r",
        allow_pickle=False,
    )
    indices = selected["aligned_face_index"].to_numpy(dtype=np.int64)
    aligned_faces = np.asarray(source_faces[indices], dtype=np.uint8)

    region_masks = None
    region_mask_uid = None
    if REGION_MASK_BUNDLE_PATH is not None:
        mask_path = Path(REGION_MASK_BUNDLE_PATH).resolve()
        with np.load(mask_path, allow_pickle=False) as bundle:
            region_masks = {
                name: np.asarray(bundle[name])
                for name in bundle.files
            }
        region_mask_uid = sha256_file(mask_path)

    model_spec_path, spec = select_model_spec(
        MODEL_REGISTRY_ROOT,
        family=MODEL_NAME,
        model_uid=prepared.model_uid,
        verify_checkpoint=True,
    )
    adapter = create_pytorch_adapter_from_spec(spec, device=DEVICE)
    result = extract_population_gradcam(
        adapter,
        aligned_faces,
        prepared,
        gradcam_batch_size=int(
            CONFIG["gradcam"]["extraction"]["gradcam_batch_size"]
        ),
        region_masks=region_masks,
        region_mask_uid=region_mask_uid,
        capture_intermediates=CAPTURE_INTERMEDIATES,
        minimum_pass_repeat_cosine=float(
            CONFIG["gradcam"]["two_pass_extraction"]["pass_b"][
                "minimum_pass_repeat_cosine"
            ]
        ),
        faithfulness_fraction=(
            float(
                CONFIG["gradcam"]["faithfulness"][
                    "primary_occlusion_fraction"
                ]
            )
            if RUN_FAITHFULNESS
            else None
        ),
        faithfulness_random_repeats=int(
            CONFIG["gradcam"]["faithfulness"]["random_repeats"]
        ),
        faithfulness_seed=SEED,
    )
    extraction_summary = {
        "selected_rows": int(len(result.features)),
        "eligible_rows": int(
            result.features["saliency_target_eligible"].sum()
        ),
        "heatmap_rows": int(len(result.heatmap_sample_ids)),
        "valid_heatmap_rows": int(
            result.features["gradcam_valid_heatmap"].fillna(False).sum()
        ),
        "saliency_spec_uid": result.saliency_spec_uid,
    }
    if extraction_summary["selected_rows"] != len(prepared.sample_ids):
        raise RuntimeError("전체 표본 feature 행이 보존되지 않았습니다.")
    if WRITE_OUTPUTS:
        if SALIENCY_ARTIFACT_OUTPUT_DIR is None:
            raise RuntimeError("SALIENCY_ARTIFACT_OUTPUT_DIR를 지정하세요.")
        write_population_saliency_artifact(
            result,
            SALIENCY_ARTIFACT_OUTPUT_DIR,
            shard_size=int(
                CONFIG["gradcam"]["extraction"]["shard_size"]
            ),
            heatmap_dtype="float16",
            overwrite=OVERWRITE,
        )
else:
    extraction_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
extraction_summary


raw activation과 gradient는 계산 중에만 사용합니다. 전체 집단에는 raw/ReLU/
normalized CAM, channel weight, scalar feature를 저장하고 full intermediate
tensor는 명시적 debug subset 외에는 저장하지 않습니다.
